In [26]:
import math
from collections import deque
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [27]:
def generate_Mongolian_Graph(n):
    A = []
    adj = defaultdict(list)
    for i in range(1,4):
        for j in range(1,n+1):
            # Checking if I should add z in the lasted
            if i == 1:
                A.append(('z', f'v{i}_{j}'))
            if j != n:
                A.append((f'v{i}_{j}', f'v{i}_{j+1}'))
            if i != 3:
                A.append((f'v{i}_{j}', f'v{i+1}_{j}'))
    for u,v in A:
      adj[u].append(v)
      adj[v].append(u)

    return adj

In [28]:
def generate_Star_Graph_1(n):
  A = [[1,2],[1,3],[1,5],[2,6],[2,10],[3,4],[3,9],[4,8],[4,10],[5,7],[5,8],[6,8],[6,9],[7,9],[10,7]]
  adj = defaultdict(list)
  for i in range(0,n):
      for u,v in A:
          u += i*9
          v += i*9
          adj[u].append(v)
          adj[v].append(u)
  return adj

In [29]:
def generate_Star_Graph_2(n):
  A = [[1,2], [1,3],[1,7], [2,8], [2,10], [3,6], [3,4], [4,10],[4,5], [5,7],[5,8],[6,9],[6,8],[7,9],[9,10]] 
  adj = defaultdict(list)
  for i in range(0,n):
      for u,v in A:
          u += i*9
          v += i*9
          adj[u].append(v)
          adj[v].append(u)
  return adj

In [30]:
def generate_Star_Graph_3(n):
  A = [[1,4],[1,7],[1,10],[2,3],[2,4],[2,8],[3,6],[3,10],[4,5],[5,6],[5,9],[6,7],[7,8],[8,9],[9,10]]
  adj = defaultdict(list)
  for i in range(0,n):
      for u,v in A:
          u += i*9
          v += i*9
          adj[u].append(v)
          adj[v].append(u)
  return adj

In [31]:
def Branch_Bound(n, graph_type = 1):
  if graph_type == 1:
      adj = generate_Mongolian_Graph(n)
      m,labels, B, used_sums = solve_min_B_BnB(adj, source='z')
  elif graph_type == 2:
      adj = generate_Star_Graph_1(n)
      m,labels, B, used_sums = solve_min_B_BnB(adj, source= 1)
  elif graph_type == 3:
      adj = generate_Star_Graph_2(n)
      m,labels, B, used_sums = solve_min_B_BnB(adj, source= 1)
  elif graph_type == 4:
      adj = generate_Star_Graph_3(n)
      m,labels, B, used_sums = solve_min_B_BnB(adj, source= 1)
  # print("Optimal B:", B)
  # print("Labels:", labels)
  # print("Used sums:", used_sums)

  def check_labeling(adj, labels):
      used = {}
      dup = []
      for u, nbrs in adj.items():
          for v in nbrs:
              if u < v:
                  s = labels[u] + labels[v]
                  if s in used:
                      dup.append(((u,v), s, used[s]))
                  else:
                      used[s] = (u,v)
      return dup

  dups = check_labeling(adj, labels)
  print("duplicates:", len(dups))
  return m,labels, B, used_sums

In [32]:
def edges_from_adj(adj):
    E = set()
    for u, nbrs in adj.items():
        for v in nbrs:
            if u < v:
                E.add((u, v))
    return sorted(E)

from collections import deque
import math

def bfs_order(adj, source):
    q = deque([source])
    seen = {source}
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    # nếu đồ thị không liên thông
    for v in adj.keys():
        if v not in seen:
            order.append(v)
    return order

def feasible_with_B_BnB(adj, B, source=1):
    if not isinstance(adj, dict):
        raise TypeError(f"adj must be a dict mapping node->list/neighbors, got {type(adj).__name__}")
    if source not in adj:
        raise ValueError(f"source {source} not in adj keys: {list(adj.keys())[:10]}...")
    if not isinstance(B, int) or B <= 0:
        raise ValueError("B must be a positive integer")

    order = bfs_order(adj, source)
    label = {v: None for v in adj}
    label[source] = 1
    used_sums = set()

    frontier = set()
    for w in adj[source]:
        frontier.add(w)

    def feasible_values_for(v):
        feas = []
        for x in range(1, B + 1):
            ok = True
            local = set()
            for w in adj[v]:
                lw = label.get(w)
                if lw is None:
                    continue
                s = x + lw
                if s in used_sums or s in local:
                    ok = False
                    break
                local.add(s)
            if ok:
                feas.append((x, local))
        return feas

    def frontier_bound_ok():
        best_v = None
        best_list = None
        for u in list(frontier):
            if label[u] is not None:
                continue
            cand = feasible_values_for(u)
            if not cand:
                return False, None, None
            if best_list is None or len(cand) < len(best_list):
                best_v, best_list = u, cand
        return True, best_v, best_list

    def next_unlabeled_nonfrontier():
        for v in order:
            if label[v] is None:
                return v
        return None

    def dfs():
        if all(label[v] is not None for v in adj):
            return True

        ok, v_mrv, cand_mrv = frontier_bound_ok()
        if not ok:
            return False

        if v_mrv is None:
            v = next_unlabeled_nonfrontier()
            if v is None:
                return True
            cand = feasible_values_for(v)
            if not cand:
                return False
            for w in adj[v]:
                if label[w] is not None:
                    frontier.add(v)
                    break
            v_pick, cand_list = v, cand
        else:
            v_pick, cand_list = v_mrv, cand_mrv

        for x, local in cand_list:
            label[v_pick] = x
            added_sums = []
            for s in local:
                used_sums.add(s)
                added_sums.append(s)

            newly_added = []
            if v_pick in frontier:
                frontier.discard(v_pick)
            for w in adj[v_pick]:
                if label[w] is None and w not in frontier:
                    frontier.add(w)
                    newly_added.append(w)

            ok2, _, _ = frontier_bound_ok()
            if ok2 and dfs():
                return True

            for s in added_sums:
                used_sums.remove(s)
            label[v_pick] = None
            for w in newly_added:
                if all(label[z] is None for z in adj[w]):
                    frontier.discard(w)
        return False

    return (label, used_sums) if dfs() else None

def solve_min_B_BnB(adj, source=1):
    if not isinstance(adj, dict):
        raise TypeError(f"adj must be a dict mapping node->list/neighbors, got {type(adj).__name__}")
    if source not in adj:
        raise ValueError(f"source {source} not in adj keys: {list(adj.keys())[:10]}...")

    m = sum(len(nbrs) for nbrs in adj.values()) // 2
    LB = math.ceil((m + 1) / 2)

    B = LB
    while True:
        res = feasible_with_B_BnB(adj, B, source=source)
        if res is not None:
            labels, used_sums = res
            return m, labels, B, used_sums
        B += 1


In [ ]:
# # # # # # # How to Use:
#  m: total number of edges in the graph
#  B: the largest label assigned in the graph
#  used_sums: the weights (sums) of the edges
#   ---- ----- ---- ----
#   This function works well for Problem 1 and Problem 2.
#   To check Problem 1, try this:
#     m, labels, B, used_sums = Branch_Bound(5, 1)  # 5 is the number of columns, and 1 defines the Mongolian tent graph
#   To check Problem 2, try this:
#     m, labels, B, used_sums = Branch_Bound(1, 2)  # 1 is the number of stars, and 2 defines the star graph
# Make sure to import the required libraries.
t0 = time.perf_counter()
m,labels, B, used_sums = Branch_Bound(10, 1)
t1 = time.perf_counter()
labels

In [ ]:
t1-t0

In [34]:
## Calculating upper, lower bound and max k of the algorithms
#length = 100
#lower_bound = []
#upper_bound = []
#algorithms = []
#for i in range(1, length,10):
#    edge,labels,B,_ = Branch_Bound(i, 2) #Be sure to check if you choose correct graph and length?
#    v = len(labels)
#    lower_bound.append((edge+1)/2)
#    upper_bound.append(edge * np.log2(v))
#    algorithms.append(B)


In [35]:
#plt.figure(figsize=(10,4))
#plt.plot(algorithms, label = 'Computer Results')
#plt.plot(lower_bound, label = 'Lower Bound (E+1/2)')
# plt.plot(upper_bound, label = 'Upper Bound (Elog2V)')
#plt.xlabel('n (Order of Graph)')
#plt.ylabel('Edge Irregularity Strength (k)')
#plt.title('Comparison of Lower Bound, Computer Results, and Upper Bound Problem 2 - Graph#2')
#plt.grid(axis= 'y')
#plt.legend()

In [36]:
import time
import tracemalloc

In [37]:
def run_test(n, source=1):
    tracemalloc.start()
    t_start = time.perf_counter()
    m, labels, B, used_sums = Branch_Bound(n, source)
    t_end = time. perf_counter()

    current, peak = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    max_label_used = max(labels.values())
    optimal_label = B
    runtime_sec = t_end-t_start
    peak_mem_mb = peak /(1024*1024)

    return max_label_used, optimal_label, runtime_sec, peak_mem_mb

def test_algo_perf(start, end, source=1):
    perf_arr = pd.DataFrame(columns=['n', 'max_label', 'optimal_max_label', 'runtime', 'memory'])
    for i in range(start, end):
        max_label, opt_label, runtime, peak_mem = run_test(i, source)
        # add row of i & outputs from run_test
        row = {'n': i, 'max_label': max_label, 'optimal_max_label': opt_label, "runtime": runtime, "memory": peak_mem}
        perf_arr.loc[len(perf_arr.index)] = row
    print(perf_arr)


#test_algo_perf(5, 11, 1)
